# Load and merge datasets

In [321]:
import pandas as pd

# Define the full paths to each CSV file
lab_1_df = pd.read_csv('processed_diamond_lab_1.csv')
lab_2_df = pd.read_csv('processed_diamond_lab_2.csv')
lab_3_df = pd.read_csv('processed_diamond_lab_3.csv')
lab_4_df = pd.read_csv('processed_diamond_lab_4.csv')

natural_1_df = pd.read_csv('processed_diamond_natural_1.csv')
natural_2_df = pd.read_csv('processed_diamond_natural_2.csv')
natural_3_df = pd.read_csv('processed_diamond_natural_3.csv')
natural_4_df = pd.read_csv('processed_diamond_natural_4.csv')


In [322]:
# Concatenate them vertically
data = pd.concat([lab_1_df, lab_2_df, lab_3_df, lab_4_df, natural_1_df, natural_2_df, natural_3_df, natural_4_df], ignore_index=True)


# Data Cleaning

In [323]:
data.shape

(64050, 21)

In [324]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64050 entries, 0 to 64049
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   index            64050 non-null  float64
 1   retailer         64050 non-null  object 
 2   lab              64050 non-null  object 
 3   shape            64050 non-null  object 
 4   polish           64050 non-null  object 
 5   symmetry         64050 non-null  float64
 6   fluorescence     64050 non-null  object 
 7   discount_%       62162 non-null  float64
 8   retail_price     64049 non-null  float64
 9   price_carat      64049 non-null  float64
 10  wire_price       64049 non-null  float64
 11  fair_price_diif  64049 non-null  float64
 12  hidden_price     64049 non-null  float64
 13  qualityScore     64049 non-null  object 
 14  qualityMaxScore  64048 non-null  float64
 15  cut              64048 non-null  object 
 16  color            64048 non-null  object 
 17  clarity     

In [325]:
data.head()

,index,retailer,lab,shape,polish,symmetry,fluorescence,discount_%,retail_price,price_carat,...,fair_price_diif,hidden_price,qualityScore,qualityMaxScore,cut,color,clarity,carat,report,price
0,3.0,Avignon Diamonds,2,Round,1,1.0,0,11.48,2842.00,1359.81,...,-64.0,5664.3730,18.0,18.0,Super Ideal,E,VVS2,2.09,IGI,11520.0
1,3.0,Cirrus Gems,2,Round,1,1.0,0,94.80,2797.00,1338.28,...,-63.0,5384.3765,18.0,18.0,Super Ideal,E,VVS2,2.09,IGI,11520.0
2,3.0,Ember Star,2,Round,1,1.0,0,13.46,2794.11,1336.89,...,-65.0,5664.1978,18.0,18.0,Super Ideal,E,VVS2,2.09,IGI,11520.0
3,3.0,Jane Brear,2,Round,1,1.0,0,11.29,2805.00,1342.11,...,-63.0,5499.7104,18.0,18.0,Super Ideal,E,VVS2,2.09,IGI,11520.0
4,3.0,KIYO,2,Round,1,1.0,0,16.26,2789.00,1334.45,...,-64.0,5599.2236,18.0,18.0,Super Ideal,E,VVS2,2.09,IGI,11520.0


In [326]:
# data.isnull().sum()

In [327]:
data.columns

Index(['index', 'retailer', 'lab', 'shape', 'polish', 'symmetry',
       'fluorescence', 'discount_%', 'retail_price', 'price_carat',
       'wire_price', 'fair_price_diif', 'hidden_price', 'qualityScore',
       'qualityMaxScore', 'cut', 'color', 'clarity', 'carat', 'report',
       'price'],
      dtype='object')

Remove Irrelevant Columns and Handle Missing Data

In [328]:
#remove unnecessary columns
data = data.drop(columns=['index','discount_%','wire_price', 'fair_price_diif', 'hidden_price'])

#drop any rows with missing values
data = data.dropna()

Clean and Standardize Data Types

In [329]:
# Clean text columns — strip spaces and standardize casing
text_columns = ['retailer', 'shape', 'cut', 'color', 'clarity', 'report']

for col in text_columns:
    data[col] = data[col].astype(str).str.strip().str.lower()

# Special casing for certain columns
data['color'] = data['color'].str.upper()
data['clarity'] = data['clarity'].str.upper()
data['report'] = data['report'].str.upper()

# Convert numeric columns
numeric_columns = ['lab', 'polish', 'symmetry', 'fluorescence', 'retail_price', 'price_carat','qualityScore', 'qualityMaxScore', 'carat', 'price']

for col in numeric_columns:
    data[col] = pd.to_numeric(data[col], errors='coerce')

# Drop unnecessary column if 'index' is just redundant
if 'index' in data.columns:
    data.drop(columns=['index'], inplace=True)



In [330]:

# data['retailer'].value_counts()
data['lab'].value_counts()
# data['shape'].value_counts()
# data['polish'].value_counts()
# data['symmetry'].value_counts()
# data['fluorescence'].value_counts()
# data['cut'].value_counts()
# data['color'].value_counts()
# data['clarity'].value_counts()
# data['report'].value_counts()

lab
0    45661
2    18366
5       21
Name: count, dtype: int64

In [331]:
#Remove rows that have a 5 in the lab column
data = data[data['lab'] != 5]

Rename and Reclassify Diamond Type Column

In [332]:
#rename the column lab to diamond_type
data.rename(columns={'lab': 'diamond_type'}, inplace=True)

# Replace 0 with 'natural' and 2 with 'lab'
data['diamond_type'] = data['diamond_type'].replace({0: 'natural', 2: 'lab'})
data['diamond_type'].value_counts()


diamond_type
natural    45661
lab        18366
Name: count, dtype: int64

Convert Categorical Codes to Readable Quality Descriptions

In [333]:
# map these values back to readable strings for polish, symmetry, and fluorescence

polish_map = {1: 'Excellent', 2: 'Very Good', 3: 'Good'}
symmetry_map = {1.0: 'Excellent', 2.0: 'Very Good', 3.0: 'Good'}
fluorescence_map = {0: 'None', 1: 'Faint', 2: 'Medium', 3: 'Strong', 4: 'Very Strong'}

data['polish'] = data['polish'].map(polish_map)
data['symmetry'] = data['symmetry'].map(symmetry_map)
data['fluorescence'] = data['fluorescence'].map(fluorescence_map)
data


,retailer,diamond_type,shape,polish,symmetry,fluorescence,retail_price,price_carat,qualityScore,qualityMaxScore,cut,color,clarity,carat,report,price
0,avignon diamonds,lab,round,Excellent,Excellent,None,2842.00,1359.81,18.0,18.0,super ideal,E,VVS2,2.09,IGI,11520.00
1,cirrus gems,lab,round,Excellent,Excellent,None,2797.00,1338.28,18.0,18.0,super ideal,E,VVS2,2.09,IGI,11520.00
2,ember star,lab,round,Excellent,Excellent,None,2794.11,1336.89,18.0,18.0,super ideal,E,VVS2,2.09,IGI,11520.00
3,jane brear,lab,round,Excellent,Excellent,None,2805.00,1342.11,18.0,18.0,super ideal,E,VVS2,2.09,IGI,11520.00
4,kiyo,lab,round,Excellent,Excellent,None,2789.00,1334.45,18.0,18.0,super ideal,E,VVS2,2.09,IGI,11520.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64045,white lily,natural,round,Excellent,Very Good,None,1671.00,2387.14,14.0,18.0,very good,F,VS2,0.70,GIA,2832.31
64046,michelle kennedy,natural,cushion,Excellent,Excellent,None,623.00,1246.00,15.0,15.0,super ideal,I,SI1,0.50,GIA,780.00
64047,helene mitchell,natural,princess,Excellent,Very Good,None,860.00,1433.33,15.0,15.0,super ideal,F,SI1,0.60,GIA,1350.00
64048,jane brear,natural,princess,Excellent,Very Good,None,1195.00,1991.67,15.0,15.0,super ideal,F,SI1,0.60,GIA,1350.00


## Create Brilliant Earth Rows Based on Unique Diamond Attributes

For each unique diamond configuration (cut, color, clarity, etc.), 
create a new row representing Brilliant Earth's price for that diamond. 
Furthermore, impute the columns: polish, symmetry, flourescence, qualityscore, qualityScoreM, and wire_price



In [334]:
import numpy as np

# Define identifying columns
id_cols = ['diamond_type', 'shape', 'cut', 'color', 'clarity', 'carat']

# Create Brilliant Earth base DataFrame
brilliant_earth_df = data.drop_duplicates(subset=id_cols).copy()
brilliant_earth_df['retailer'] = 'brilliant earth'
brilliant_earth_df['retail_price'] = brilliant_earth_df['price']
brilliant_earth_df['price_carat'] = brilliant_earth_df['retail_price'] / brilliant_earth_df['carat']

# Remove `price` column from both datasets
brilliant_earth_df = brilliant_earth_df.drop(columns=['price'])
data = data.drop(columns=['price'])

# Define the columns to impute
most_common_cols = ['polish', 'symmetry', 'flourescence']
mean_cols = ['qualityScore', 'qualityScoreM', 'wire_price']

# Merge for imputing most common values
def impute_mode(col):
    mode_df = (
        data
        .groupby(id_cols)[col]
        .agg(lambda x: x.mode().iloc[0] if not x.mode().dropna().empty else np.nan)
        .reset_index()
    )
    
    # Rename to avoid collision
    mode_df = mode_df.rename(columns={col: f"{col}_imputed"})

    # Merge and extract
    merged = brilliant_earth_df.merge(mode_df, on=id_cols, how='left')

    if f"{col}_imputed" in merged.columns:
        return merged[f"{col}_imputed"]
    else:
        return pd.Series([np.nan] * len(brilliant_earth_df), index=brilliant_earth_df.index)

# Impute polish, symmetry and flourescence
for col in most_common_cols:
    if col in data.columns:
        brilliant_earth_df[col] = impute_mode(col).values

# Merge for imputing average values
def impute_mean(col):
    mean_df = (
        data
        .groupby(id_cols)[col]
        .mean()
        .reset_index()
        .rename(columns={col: f"{col}_imputed"})
    )
    
    merged = brilliant_earth_df.merge(mean_df, on=id_cols, how='left')

    if f"{col}_imputed" in merged.columns:
        return merged[f"{col}_imputed"]
    else:
        return pd.Series([np.nan] * len(brilliant_earth_df), index=brilliant_earth_df.index)

# Impute qualityScore, qualityMaxScore, wire_price
for col in mean_cols:
    if col in data.columns:
        values = impute_mean(col)
        brilliant_earth_df[col] = values.values

# Final concat
data = pd.concat([data, brilliant_earth_df], ignore_index=True)

In [335]:
data

,retailer,diamond_type,shape,polish,symmetry,fluorescence,retail_price,price_carat,qualityScore,qualityMaxScore,cut,color,clarity,carat,report
0,avignon diamonds,lab,round,Excellent,Excellent,None,2842.00,1359.810000,18.000000,18.0,super ideal,E,VVS2,2.09,IGI
1,cirrus gems,lab,round,Excellent,Excellent,None,2797.00,1338.280000,18.000000,18.0,super ideal,E,VVS2,2.09,IGI
2,ember star,lab,round,Excellent,Excellent,None,2794.11,1336.890000,18.000000,18.0,super ideal,E,VVS2,2.09,IGI
3,jane brear,lab,round,Excellent,Excellent,None,2805.00,1342.110000,18.000000,18.0,super ideal,E,VVS2,2.09,IGI
4,kiyo,lab,round,Excellent,Excellent,None,2789.00,1334.450000,18.000000,18.0,super ideal,E,VVS2,2.09,IGI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75029,brilliant earth,natural,round,Excellent,Excellent,None,4258.33,5193.085366,18.000000,18.0,super ideal,G,VVS2,0.82,GIA
75030,brilliant earth,natural,emerald,Excellent,Excellent,None,2070.00,3696.428571,15.000000,15.0,ideal,D,IF,0.56,GIA
75031,brilliant earth,natural,round,Excellent,Very Good,Medium,2832.31,4046.157143,14.363636,18.0,very good,F,VS2,0.70,GIA
75032,brilliant earth,natural,cushion,Excellent,Excellent,None,780.00,1560.000000,15.000000,15.0,super ideal,I,SI1,0.50,GIA


**diamond_type**: Indicates whether the diamond is lab (lab-grown) or natural.

**shape**: The physical shape of the diamond (e.g., round, oval, pear, emerald).

**cut**: Quality of the diamond cut, influencing brilliance (e.g., ideal, super ideal, very good).

**color**: Diamond color grade ranging from D (most colorless) to J and lower.

**clarity**: Measures internal/external flaws; grades like FL, IF, VVS1, VS1, SI1.

**carat**: Weight of the diamond. Heavily impacts pricing.

**polish**: Smoothness of the diamond’s surface; usually Excellent, Very Good, or Good.

**symmetry**: How well-aligned and proportioned the diamond’s facets are.

**fluorescence**: How much the diamond glows under UV light (e.g., None, Faint, Strong).

**report**: Certification lab or authority (e.g., GIA, IGI, GCAL).

**retail_price**: The listed price by the retailer. (Brilliant Earth's price is from 2018, the rest is from 2025)

**price_carat**: Price per carat.

**qualityScore**:Quality rating based on the diamond’s attributes.

**qualityMaxScore**: The maximum possible quality score for this diamond configuration.

##  Adjust Brilliant Earth Prices to 2025

Create `price_2025` and `price_carat_2025` columns to reflect adjusted 2025 values for Brilliant Earth, using the computed indices, keeping other retailers' prices unchanged.

In [343]:
#Make a copy of the dataframe
df = data.copy()

# Compute average price per carat in 2018 (Brilliant Earth)
avg_2018 = (df[df['retailer'] == 'brilliant earth']['retail_price'] / 
            df[df['retailer'] == 'brilliant earth']['carat']).mean()

# Compute average price per carat in 2025 (non-Brilliant Earth)
avg_2025 = df[df['retailer'] != 'brilliant earth']['price_carat'].mean()

# Calculate index ratio
diamond_index_2025 = (avg_2025 / avg_2018) * 100

# Create the new column, keeping all prices the same except for Brilliant Earth
df['price_2025'] = df.apply(
    lambda row: round(row['retail_price'] * (diamond_index_2025 / 100), 2)
    if row['retailer'] == 'brilliant earth'
    else row['retail_price'],
    axis=1)

#Adjust price per carat
df['price_carat_2025'] = df.apply(
    lambda row: round(row['price_carat'] * (diamond_index_2025 / 100), 2)
    if row['retailer'] == 'brilliant earth'
    else row['price_carat'],
    axis=1
)
print(f"Diamond Price Index 2025: {diamond_index_2025:.2f}")
df

Diamond Price Index 2025: 66.86


,retailer,diamond_type,shape,polish,symmetry,fluorescence,retail_price,price_carat,qualityScore,qualityMaxScore,cut,color,clarity,carat,report,price_2025,price_carat_2025
0,avignon diamonds,lab,round,Excellent,Excellent,None,2842.00,1359.810000,18.000000,18.0,super ideal,E,VVS2,2.09,IGI,2842.00,1359.81
1,cirrus gems,lab,round,Excellent,Excellent,None,2797.00,1338.280000,18.000000,18.0,super ideal,E,VVS2,2.09,IGI,2797.00,1338.28
2,ember star,lab,round,Excellent,Excellent,None,2794.11,1336.890000,18.000000,18.0,super ideal,E,VVS2,2.09,IGI,2794.11,1336.89
3,jane brear,lab,round,Excellent,Excellent,None,2805.00,1342.110000,18.000000,18.0,super ideal,E,VVS2,2.09,IGI,2805.00,1342.11
4,kiyo,lab,round,Excellent,Excellent,None,2789.00,1334.450000,18.000000,18.0,super ideal,E,VVS2,2.09,IGI,2789.00,1334.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75029,brilliant earth,natural,round,Excellent,Excellent,None,4258.33,5193.085366,18.000000,18.0,super ideal,G,VVS2,0.82,GIA,2847.21,3472.21
75030,brilliant earth,natural,emerald,Excellent,Excellent,None,2070.00,3696.428571,15.000000,15.0,ideal,D,IF,0.56,GIA,1384.05,2471.51
75031,brilliant earth,natural,round,Excellent,Very Good,Medium,2832.31,4046.157143,14.363636,18.0,very good,F,VS2,0.70,GIA,1893.74,2705.35
75032,brilliant earth,natural,cushion,Excellent,Excellent,None,780.00,1560.000000,15.000000,15.0,super ideal,I,SI1,0.50,GIA,521.52,1043.05


# Separate the dataset into lab-grown and natural diamonds and adjust Brilliant Earth prices to 2025

Separate the dataset into lab-grown and natural diamonds, then compute a price index for each type based on Brilliant Earth's 2018 pricing and the average 2025 market prices.  
Apply this index to adjust only Brilliant Earth’s prices (`retail_price` and `price_carat`) to reflect estimated 2025 values, while leaving all other retailers’ prices unchanged.

In [338]:
#Make a copy of the dataframe
df2 = data.copy()

# Create 2 separate data frame (lab-grown vs natural)
df_lab = df2[df2['diamond_type'] == 'lab']
df_natural = df2[df2['diamond_type'] == 'natural']

#  Compute diamond price index for EACH Type

# Lab-grown
avg_2018_lab = (df_lab[df_lab['retailer'] == 'brilliant earth']['retail_price'] / 
                df_lab[df_lab['retailer'] == 'brilliant earth']['carat']).mean()

avg_2025_lab = df_lab[df_lab['retailer'] != 'brilliant earth']['price_carat'].mean()

index_2025_lab = (avg_2025_lab / avg_2018_lab) * 100

# Natural
avg_2018_natural = (df_natural[df_natural['retailer'] == 'brilliant earth']['retail_price'] / 
                    df_natural[df_natural['retailer'] == 'brilliant earth']['carat']).mean()

avg_2025_natural = df_natural[df_natural['retailer'] != 'brilliant earth']['price_carat'].mean()

index_2025_natural = (avg_2025_natural / avg_2018_natural) * 100

# Adjust brilliant earth prices

#  Lab-grown Diamond Index
df_lab['price_2025'] = df_lab.apply(
    lambda row: round(row['retail_price'] * (index_2025_lab / 100), 2)
    if row['retailer'] == 'brilliant earth' else row['retail_price'],
    axis=1
)

df_lab['price_carat_2025'] = df_lab.apply(
    lambda row: round(row['price_carat'] * (index_2025_lab / 100), 2)
    if row['retailer'] == 'brilliant earth' else row['price_carat'],
    axis=1
)

# Natural Diamond Index
df_natural['price_2025'] = df_natural.apply(
    lambda row: round(row['retail_price'] * (index_2025_natural / 100), 2)
    if row['retailer'] == 'brilliant earth' else row['retail_price'],
    axis=1
)

df_natural['price_carat_2025'] = df_natural.apply(
    lambda row: round(row['price_carat'] * (index_2025_natural / 100), 2)
    if row['retailer'] == 'brilliant earth' else row['price_carat'],
    axis=1
)


/var/folders/7t/_53jmnd13dz0gskj6w78cr5m0000gn/T/ipykernel_48088/2810186556.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lab['price_2025'] = df_lab.apply(
/var/folders/7t/_53jmnd13dz0gskj6w78cr5m0000gn/T/ipykernel_48088/2810186556.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_lab['price_carat_2025'] = df_lab.apply(
/var/folders/7t/_53jmnd13dz0gskj6w78cr5m0000gn/T/ipykernel_48088/2810186556.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.

In [339]:
print(f"Lab-Grown Diamond Price Index 2025: {index_2025_lab:.2f}")
print(f"Natural Diamond Price Index 2025: {index_2025_natural:.2f}")

Lab-Grown Diamond Price Index 2025: 37.63
Natural Diamond Price Index 2025: 74.31


In [340]:
df_lab.head()

,retailer,diamond_type,shape,polish,symmetry,fluorescence,retail_price,price_carat,qualityScore,qualityMaxScore,cut,color,clarity,carat,report,price_2025,price_carat_2025
0,avignon diamonds,lab,round,Excellent,Excellent,None,2842.00,1359.81,18.0,18.0,super ideal,E,VVS2,2.09,IGI,2842.00,1359.81
1,cirrus gems,lab,round,Excellent,Excellent,None,2797.00,1338.28,18.0,18.0,super ideal,E,VVS2,2.09,IGI,2797.00,1338.28
2,ember star,lab,round,Excellent,Excellent,None,2794.11,1336.89,18.0,18.0,super ideal,E,VVS2,2.09,IGI,2794.11,1336.89
3,jane brear,lab,round,Excellent,Excellent,None,2805.00,1342.11,18.0,18.0,super ideal,E,VVS2,2.09,IGI,2805.00,1342.11
4,kiyo,lab,round,Excellent,Excellent,None,2789.00,1334.45,18.0,18.0,super ideal,E,VVS2,2.09,IGI,2789.00,1334.45


In [341]:
df_natural.head()

,retailer,diamond_type,shape,polish,symmetry,fluorescence,retail_price,price_carat,qualityScore,qualityMaxScore,cut,color,clarity,carat,report,price_2025,price_carat_2025
17864,crown jewels,natural,round,Excellent,Excellent,None,474.0,1823.08,18.0,18.0,super ideal,E,VS2,0.26,GIA,474.0,1823.08
17865,eccles diamonds,natural,round,Excellent,Excellent,None,481.0,1850.00,18.0,18.0,super ideal,E,VS2,0.26,GIA,481.0,1850.00
17866,jane brear,natural,round,Excellent,Excellent,None,915.0,3519.23,18.0,18.0,super ideal,E,VS2,0.26,GIA,915.0,3519.23
17867,michelle kennedy,natural,round,Very Good,Excellent,None,881.0,3388.46,18.0,18.0,super ideal,E,VS2,0.26,GIA,881.0,3388.46
17868,temperley,natural,round,Excellent,Excellent,None,915.0,3519.23,18.0,18.0,super ideal,E,VS2,0.26,GIA,915.0,3519.23


# Save Final DataFrames to CSV

In [342]:
#  Save Each DataFrame as a CSV
df.to_csv("df.csv", index=False)
df_lab.to_csv("df_lab.csv", index=False)
df_natural.to_csv("df_natural.csv", index=False)
